# Template 04: Model Preparation

**Purpose:** Prepare data for XGBoost training

**Inputs:**
- data/02_conditioned.parquet

**Outputs:**
- data/04_train.parquet
- data/04_test.parquet

In [1]:
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_coll/v1"


In [3]:
import pandas as pd
import yaml
import os
import sys
import gc
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 04: MODEL PREPARATION")
print("########################################")

project_root = setup_notebook_environment()

########################################
# STAGE 04: MODEL PREPARATION
########################################


In [4]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']

In [5]:
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
data = pd.read_parquet(checkpoint_02)

# Validate critical columns exist
critical_cols = [
    cfg['data']['join_key'],
    cfg['data']['fold_column'],
    cfg['experiment']['target'],
    cfg['experiment']['exposure']
]
missing = [c for c in critical_cols if c not in data.columns]
if missing:
    raise ValueError(f"Missing critical columns: {missing}")

print(f"\n* Loaded: {data.shape}")
print(f"* Critical columns validated: {', '.join(critical_cols)}")


* Loaded: (5424070, 112)
* Critical columns validated: vin_date, fold, pp_bi, ee_bi_imps


In [6]:
# Split by fold
fold_col = cfg['data']['fold_column']
test_fold = cfg['debug']['test_fold']
train_folds = cfg['debug']['fold_mapping'][cfg['debug']['level']]

train_data = data[data[fold_col].isin(train_folds)].copy()
test_data = data[data[fold_col] == test_fold].copy()

print(f"\n* Train: {train_data.shape}")
print(f"* Test: {test_data.shape}")

del data
gc.collect()


* Train: (2716120, 112)
* Test: (2707950, 112)


0

In [7]:
# Validate outputs before saving
from utils import validate_critical_columns
validate_critical_columns(train_data, cfg, 'Stage 04 Train Output')
validate_critical_columns(test_data, cfg, 'Stage 04 Test Output')

# Save splits
train_file = f"{output_base}/data/04_train.parquet"
test_file = f"{output_base}/data/04_test.parquet"

train_data.to_parquet(train_file)
test_data.to_parquet(test_file)

print(f"\n* Saved: {train_file}")
print(f"* Saved: {test_file}")

[Stage 04 Train Output] ✓ Critical columns validated: vin_date, fold, pp_bi, ee_bi_imps
[Stage 04 Test Output] ✓ Critical columns validated: vin_date, fold, pp_bi, ee_bi_imps



* Saved: output/car_coll/v1/data/04_train.parquet
* Saved: output/car_coll/v1/data/04_test.parquet


In [8]:
print("\n########################################")
print("# STAGE 04: COMPLETE")
print("########################################")


########################################
# STAGE 04: COMPLETE
########################################
